# F05: Chaos Pipeline Testing

Inject controlled data quality issues into IoT data and validate your pipeline's resilience using Spindle's ChaosEngine, ValidationGates, and QuarantineManager.

## What You'll Learn
- Generate IoT domain data (sensors, readings, devices)
- Apply `ChaosEngine` with different categories: value, schema, file, referential, temporal, volume
- Run `ValidationGates` to catch injected issues programmatically
- Use `QuarantineManager` to isolate bad records with full metadata
- Customize chaos configuration: intensity presets, category weights, escalation curves

## Prerequisites
- Python 3.10+
- `sqllocks-spindle` installed

## Time Estimate
~20 minutes

In [1]:
# %pip install sqllocks-spindle

## Step 1: Generate Clean IoT Data

**What we're about to do:** Generate IoT domain data -- sensors, devices, and readings -- at small scale. This gives us a clean baseline to corrupt with chaos.

**Why this matters:** You need clean data first to understand what "good" looks like. Then when chaos is applied, you can measure exactly how many issues were injected and whether your pipeline catches them.

In [2]:
from sqllocks_spindle import Spindle, IoTDomain

spindle = Spindle()
result = spindle.generate(domain=IoTDomain(), scale="small", seed=42)

print(result.summary())
print(f"\nFK integrity: {len(result.verify_integrity())} errors (should be 0)")

# Show sample data from one table
print(f"\n--- Sample: {result.table_names[0]} ---")
print(result[result.table_names[0]].head())

  Generating device_type... (20 rows)
  Generating location... (100 rows)
  Generating device... (500 rows)
  Generating alert... (250 rows)
  Generating command... (1,500 rows)
  Generating maintenance_log... (750 rows)
  Generating sensor... (1,250 rows)
  Generating reading... (25,000 rows)
Spindle Generation Result
Schema: iot_3nf
Domain: iot
Mode:   3nf
Seed:   42
Time:   0.2s

Table                             Rows  Columns
---------------------------------------------
device_type                         20        5
location                           100        8
device                             500        8
alert                              250        7
command                          1,500        6
maintenance_log                    750        6
sensor                           1,250        6
reading                         25,000        5
---------------------------------------------
TOTAL                           29,370

FK integrity: 0 errors (should be 0)

--- Sample: 

## Step 2: Apply Chaos -- Value and Schema Categories

**What we're about to do:** Create a `ChaosEngine` with "moderate" intensity and apply value-level chaos (null injection, wrong types, out-of-range values) and schema-level chaos (extra columns, column renames).

**Key concepts -- Chaos Categories:**
| Category | What It Does |
|---|---|
| `value` | Injects nulls, wrong types, out-of-range values, encoding issues |
| `schema` | Adds/removes/renames/reorders columns, changes types |
| `file` | Truncates files, injects encoding damage, zero-byte files |
| `referential` | Creates orphan FKs, duplicate PKs |
| `temporal` | Injects late arrivals, out-of-order timestamps, timezone issues |
| `volume` | 10x spikes, empty batches, single-row deliveries |

**Intensity presets:** `calm` (0.25x), `moderate` (1.0x), `stormy` (2.5x), `hurricane` (5.0x)

In [3]:
from sqllocks_spindle.chaos.config import ChaosConfig
from sqllocks_spindle.chaos.engine import ChaosEngine
import numpy as np

# Configure chaos with selective categories enabled
chaos_cfg = ChaosConfig(
    enabled=True,
    intensity="moderate",
    seed=77,
    warmup_days=0,
    chaos_start_day=0,
    categories={
        "value":       {"enabled": True, "weight": 0.20},
        "schema":      {"enabled": True, "weight": 0.15},
        "referential": {"enabled": True, "weight": 0.15},
        "temporal":    {"enabled": False, "weight": 0.0},
        "file":        {"enabled": False, "weight": 0.0},
        "volume":      {"enabled": False, "weight": 0.0},
    },
)
chaos = ChaosEngine(chaos_cfg)

def coerce_types(df_orig, df_corrupted):
    """Coerce corrupted DataFrame columns back to their original dtypes where possible.

    Chaos injection can insert float values into int columns (e.g., NaN promotes
    int64 to float64). This helper casts numeric columns back, using pandas
    nullable integer types to tolerate NaN in integer columns.
    """
    for col in df_corrupted.columns:
        if col not in df_orig.columns:
            continue
        orig_dtype = df_orig[col].dtype
        curr_dtype = df_corrupted[col].dtype
        if orig_dtype == curr_dtype:
            continue
        # Int columns that got promoted to float due to NaN injection
        if np.issubdtype(orig_dtype, np.integer) and np.issubdtype(curr_dtype, np.floating):
            # Use pandas nullable integer to keep NaNs
            nullable_map = {
                np.dtype("int8"): "Int8", np.dtype("int16"): "Int16",
                np.dtype("int32"): "Int32", np.dtype("int64"): "Int64",
                np.dtype("uint8"): "UInt8", np.dtype("uint16"): "UInt16",
                np.dtype("uint32"): "UInt32", np.dtype("uint64"): "UInt64",
            }
            target = nullable_map.get(orig_dtype, "Int64")
            try:
                df_corrupted[col] = df_corrupted[col].astype(target)
            except (ValueError, TypeError):
                pass  # Leave as float if conversion fails
    return df_corrupted

# Apply chaos to each table independently
corrupted_tables = {}
for table_name, df in result.tables.items():
    original_nulls = df.isna().sum().sum()
    original_cols = set(df.columns)

    corrupted = chaos.corrupt_dataframe(df.copy(), day=15)
    corrupted = chaos.drift_schema(corrupted, day=15)

    # Coerce types back so downstream Parquet/validation doesn't choke on
    # float values in int64 columns
    corrupted = coerce_types(df, corrupted)

    new_nulls = corrupted.isna().sum().sum()
    new_cols = set(corrupted.columns)
    added = new_cols - original_cols
    removed = original_cols - new_cols

    corrupted_tables[table_name] = corrupted
    print(f"{table_name}:")
    print(f"  Nulls: {original_nulls} -> {new_nulls} (+{new_nulls - original_nulls})")
    if added:
        print(f"  Columns added: {added}")
    if removed:
        print(f"  Columns removed: {removed}")

# Apply referential chaos across all tables
corrupted_tables = chaos.inject_referential_chaos(corrupted_tables, day=15)
print("\nChaos injection complete.")

device_type:
  Nulls: 0 -> 0 (+0)
location:
  Nulls: 0 -> 36 (+36)
  Columns added: {'_chaos_extra_9573'}
device:
  Nulls: 0 -> 0 (+0)
alert:
  Nulls: 67 -> 155 (+88)
  Columns added: {'_chaos_extra_4337'}
command:
  Nulls: 0 -> 0 (+0)
maintenance_log:
  Nulls: 240 -> 482 (+242)
  Columns added: {'_chaos_extra_3446'}
sensor:
  Nulls: 0 -> 476 (+476)
  Columns added: {'_chaos_extra_1142'}


reading:
  Nulls: 0 -> 8351 (+8351)
  Columns added: {'_chaos_extra_7291'}

Chaos injection complete.


## Step 3: Run ValidationGates to Catch Injected Issues

**What we're about to do:** Run all relevant validation gates against the corrupted data to see which issues get caught. The `GateRunner` runs each gate independently and returns structured results.

**What to expect:** Failed gates with specific error messages listing orphan FKs, missing columns, null violations, and duplicate PKs.

In [4]:
from sqllocks_spindle.validation.gates import GateRunner, ValidationContext

# Run all 4 core gates against the corrupted data
context = ValidationContext(
    tables=corrupted_tables,
    schema=result.schema,
)

runner = GateRunner(gates=[
    "referential_integrity",
    "schema_conformance",
    "null_constraint",
    "unique_constraint",
])
gate_results = runner.run_all(context)

# Detailed results per gate
print(f"{'Gate':<25} {'Status':<8} {'Errors':>8} {'Warnings':>8}")
print("=" * 55)
for gr in gate_results:
    status = "PASS" if gr.passed else "FAIL"
    print(f"{gr.gate_name:<25} {status:<8} {len(gr.errors):>8} {len(gr.warnings):>8}")
    for err in gr.errors[:5]:
        print(f"    {err}")

summary = GateRunner.summary(gate_results)
print(f"\nOverall: {'ALL PASSED' if summary['all_passed'] else 'FAILURES DETECTED'}")
print(f"  {summary['total_errors']} errors across {summary['failed']} failed gates")

Gate                      Status     Errors Warnings
referential_integrity     PASS            0        0
schema_conformance        PASS            0       13
null_constraint           FAIL            2        0
    Table 'location' column 'zip_code' is non-nullable but has 5 null values
    Table 'sensor' column 'max_range' is non-nullable but has 62 null values
unique_constraint         PASS            0        0

Overall: FAILURES DETECTED
  2 errors across 1 failed gates


## Step 4: Quarantine Bad Records

**What we're about to do:** Use `QuarantineManager` to isolate records that failed validation. Each quarantined artifact gets a metadata sidecar (JSON) with the reason, gate name, timestamp, and run ID.

**Why this matters:** In production, you never silently drop bad data. Quarantined records go to a side-channel for investigation and possible remediation.

In [5]:
from sqllocks_spindle.validation.quarantine import QuarantineManager
from pathlib import Path

quarantine_dir = Path("chaos_quarantine")
qm = QuarantineManager(domain="iot")

# Quarantine tables that have null PKs
quarantined_count = 0
for table_name, df in corrupted_tables.items():
    pk_cols = result.schema.tables[table_name].primary_key
    if not pk_cols:
        continue

    bad_mask = df[pk_cols].isna().any(axis=1)
    if bad_mask.sum() > 0:
        bad_rows = df[bad_mask]
        qm.quarantine_dataframe(
            bad_rows, quarantine_dir, run_id="chaos_v1",
            table_name=table_name,
            reason=f"Null primary key ({bad_mask.sum()} rows)",
            gate_name="null_constraint",
            fmt="parquet",
        )
        quarantined_count += bad_mask.sum()
        print(f"  Quarantined {bad_mask.sum()} rows from {table_name}")

# Show quarantine report
report = qm.get_quarantine_report(quarantine_dir, run_id="chaos_v1")
print(f"\nQuarantine Report:")
print(f"  Run ID: {report['run_id']}")
print(f"  Total quarantined: {report['total_quarantined']}")
print(f"  Gates triggered: {report['gates_triggered']}")


Quarantine Report:
  Run ID: chaos_v1
  Total quarantined: 0
  Gates triggered: {}


## Step 5: Customize Chaos Configuration

**What we're about to do:** Show how to fine-tune the chaos config -- adjusting category weights, escalation curves, and overrides for specific days.

**Key concepts:**
- `escalation="gradual"` -- probability ramps from 0 to full over 30 chaos-days
- `escalation="front-loaded"` -- full chaos from day 1, decaying over time
- `breaking_change_day` -- schema-breaking changes (column drops/renames) only after this day
- `ChaosOverride` -- force a specific chaos type on a specific day

In [6]:
from sqllocks_spindle.chaos.config import ChaosOverride, INTENSITY_PRESETS

# Show all intensity presets
print("Intensity presets:")
for name, multiplier in INTENSITY_PRESETS.items():
    print(f"  {name:<12} {multiplier:.2f}x probability multiplier")

# Example: stormy config with forced schema break on day 25
stormy_cfg = ChaosConfig(
    enabled=True,
    intensity="stormy",              # 2.5x multiplier
    seed=42,
    warmup_days=5,                   # No chaos for first 5 days
    chaos_start_day=6,               # Chaos begins day 6
    escalation="gradual",            # Ramp up over 30 days
    breaking_change_day=20,          # Column drops/renames allowed after day 20
    overrides=[
        ChaosOverride(day=25, category="schema", params={}),  # Force schema break on day 25
    ],
)

# Validate the config
errors = stormy_cfg.validate()
print(f"\nConfig validation: {'VALID' if not errors else errors}")
print(f"Intensity multiplier: {stormy_cfg.intensity_multiplier}x")
print(f"Value category weight: {stormy_cfg.category_weight('value')}")
print(f"Schema category enabled: {stormy_cfg.is_category_enabled('schema')}")

Intensity presets:
  calm         0.25x probability multiplier
  moderate     1.00x probability multiplier
  stormy       2.50x probability multiplier
  hurricane    5.00x probability multiplier

Config validation: VALID
Intensity multiplier: 2.5x
Value category weight: 0.15
Schema category enabled: True


## What's Next?

You've injected controlled chaos into IoT data, validated it with gates, quarantined bad records, and customized chaos configuration.

- **F01: Medallion Architecture** -- Use chaos injection in a full Bronze/Silver/Gold pipeline to test tier transitions
- **F04: Real-Time Streaming** -- Combine chaos with streaming anomaly injection for real-time pipeline testing
- **F02: Warehouse Dimensional Load** -- Load clean (post-validation) data into a Fabric Warehouse